### 1. DP1 ECDFS run inventory notebook

This notebook collects the information needed to document the completed DP1 ECDFS run of the LSST--VISTA workflow.

It is not a science-validation notebook. Its purpose is to make a compact inventory of the run products for users who do not know where the products are stored or what was produced. The output can be used to write a short DP1 run note describing the Butler repository, collections, coadd-image footprint, forced-catalogue footprint, ComCam coadd-image location, and final reduced catalogue.


### 2. Imports, paths, and run settings

In [2]:
from pathlib import Path
import re
import json
from datetime import datetime

import numpy as np
import pandas as pd

try:
    from astropy.table import Table
except Exception:
    Table = None

try:
    import lsst.daf.butler as dafButler
    HAS_LSST = True
except Exception as e:
    HAS_LSST = False
    print("LSST stack is not available in this kernel.")
    print("The Butler inventory sections require an LSST environment.")
    print(type(e).__name__, e)

# ------------------------------------------------------------------
# Main locations
# ------------------------------------------------------------------

# Butler repository for the DP1 ECDFS run.
BUTLER_REPO = Path("../../../dmu4/dmu4_DP1/dmu4_DP1_ECDFS/data")

# Final reduced catalogue used by the photometry and astrometry notebooks.
DATA_DIR = Path("../data")
FINAL_REDUCED_CATALOGUE = DATA_DIR / "full_reduced_cat_DP1_20260408.fits"

# ComCam coadd images were stored as FITS files under the Butler data directory.
# This is the path that should be mentioned in the DP1 run note.
COMCAM_CALEXP_DIR = BUTLER_REPO / "ComCam/deepCoadd_results/deepCoadd_calexp"

# ------------------------------------------------------------------
# Butler collections
# ------------------------------------------------------------------

SKYMAP = "lsst_cells_v1"

VIRCAM_CALEXP_COLLECTION = "u/ir-sare1/DRP/videoCoaddDetect"
FORCED_COLLECTION = "u/ir-sare1/DRP/videoMultiVisit/20260406T094204Z"

# Optional: set this if a measured-catalogue collection exists.
# If not available, leave it equal to FORCED_COLLECTION; the notebook will
# simply report no deepCoadd_meas refs if none are present.
MEAS_COLLECTION = FORCED_COLLECTION

# ------------------------------------------------------------------
# Bands
# ------------------------------------------------------------------

COMCAM_BANDS = ["u", "g", "r", "i", "z", "y"]
VIRCAM_BANDS = ["Z", "Y", "J", "H", "K"]
ALL_BANDS = COMCAM_BANDS + VIRCAM_BANDS

# ------------------------------------------------------------------
# Runtime options
# ------------------------------------------------------------------

# Counting rows in every forced catalogue requires loading each catalogue.
# Keep this True for the final inventory run. Set False if only footprint
# counts are needed.
COUNT_FORCED_ROWS = False

# A forced catalogue with fewer rows than this will be listed as low-row.
LOW_ROW_THRESHOLD = 10

OUTDIR = Path("data/dp1_run_inventory")
OUTDIR.mkdir(parents=True, exist_ok=True)

print("Butler repo:", BUTLER_REPO)
print("ComCam coadd FITS directory:", COMCAM_CALEXP_DIR)
print("VIRCAM coadd collection:", VIRCAM_CALEXP_COLLECTION)
print("Forced-catalogue collection:", FORCED_COLLECTION)
print("Measured-catalogue collection:", MEAS_COLLECTION)
print("Final reduced catalogue:", FINAL_REDUCED_CATALOGUE)
print("Output directory:", OUTDIR)

Butler repo: ../../../dmu4/dmu4_DP1/dmu4_DP1_ECDFS/data
ComCam coadd FITS directory: ../../../dmu4/dmu4_DP1/dmu4_DP1_ECDFS/data/ComCam/deepCoadd_results/deepCoadd_calexp
VIRCAM coadd collection: u/ir-sare1/DRP/videoCoaddDetect
Forced-catalogue collection: u/ir-sare1/DRP/videoMultiVisit/20260406T094204Z
Measured-catalogue collection: u/ir-sare1/DRP/videoMultiVisit/20260406T094204Z
Final reduced catalogue: ../data/full_reduced_cat_DP1_20260408.fits
Output directory: data/dp1_run_inventory


### 3. Open the Butler repository

This cell opens the DP1 ECDFS Butler repository. The notebook can still parse the ComCam FITS directory without Butler, but the VIRCAM coadd and forced-catalogue inventory require the LSST stack.

In [3]:
if not HAS_LSST:
    raise RuntimeError("Please run this notebook in an LSST stack environment.")

butler = dafButler.Butler(str(BUTLER_REPO))
registry = butler.registry

print("Opened Butler repository successfully.")
print("Registry:", registry)

Opened Butler repository successfully.
Registry: <lsst.daf.butler._registry_shim.RegistryShim object at 0x1476d296f140>


### 4. Helper functions for Butler refs and tables

In [4]:
def dataid_value(data_id, key, default=None):
    try:
        return data_id[key]
    except Exception:
        return default


def ref_to_row(ref, dataset_type, source, storage="Butler"):
    return {
        "source": source,
        "storage": storage,
        "dataset_type": dataset_type,
        "run": getattr(ref, "run", None),
        "skymap": dataid_value(ref.dataId, "skymap"),
        "tract": dataid_value(ref.dataId, "tract"),
        "patch": dataid_value(ref.dataId, "patch"),
        "band": dataid_value(ref.dataId, "band"),
        "physical_filter": dataid_value(ref.dataId, "physical_filter"),
    }


def query_dataset_refs(dataset_type, collections, source_label):
    try:
        refs = list(
            registry.queryDatasets(
                dataset_type,
                collections=collections,
                where="skymap = SKYMAP",
                bind={"SKYMAP": SKYMAP},
            )
        )
    except Exception as e:
        print(f"Query with skymap constraint failed for {dataset_type}: {type(e).__name__}: {e}")
        print("Retrying without the skymap constraint.")
        refs = list(
            registry.queryDatasets(
                dataset_type,
                collections=collections,
            )
        )

    rows = [ref_to_row(ref, dataset_type, source_label) for ref in refs]
    df = pd.DataFrame(rows)

    if len(df) > 0:
        for col in ["tract", "patch"]:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    return refs, df


def clean_inventory_df(df):
    if len(df) == 0:
        return df

    cols = [
        "source", "storage", "dataset_type", "run", "skymap",
        "tract", "patch", "band", "physical_filter",
    ]

    for col in cols:
        if col not in df.columns:
            df[col] = None

    return df[cols].sort_values(["dataset_type", "tract", "patch", "band"], na_position="last")


def count_by_tract_band(df):
    if len(df) == 0:
        return pd.DataFrame()

    use = df.dropna(subset=["tract", "patch", "band"]).copy()
    if len(use) == 0:
        return pd.DataFrame()

    use["tract"] = use["tract"].astype(int)
    use["patch"] = use["patch"].astype(int)

    tab = (
        use.groupby(["tract", "band"])["patch"]
        .nunique()
        .reset_index(name="n_patches")
        .pivot(index="tract", columns="band", values="n_patches")
        .fillna(0)
        .astype(int)
        .reset_index()
    )

    ordered_cols = ["tract"] + [b for b in ALL_BANDS if b in tab.columns]
    other_cols = [c for c in tab.columns if c not in ordered_cols]
    return tab[ordered_cols + other_cols]


def patch_units_by_tract(df):
    if len(df) == 0:
        return pd.DataFrame(columns=["tract", "n_patches"])

    use = df.dropna(subset=["tract", "patch"]).copy()
    if len(use) == 0:
        return pd.DataFrame(columns=["tract", "n_patches"])

    use["tract"] = use["tract"].astype(int)
    use["patch"] = use["patch"].astype(int)

    out = (
        use.drop_duplicates(["tract", "patch"])
        .groupby("tract")
        .size()
        .reset_index(name="n_patches")
        .sort_values("tract")
    )

    total = pd.DataFrame({"tract": ["Total"], "n_patches": [int(out["n_patches"].sum())]})
    return pd.concat([out, total], ignore_index=True)


def save_table(df, name):
    path = OUTDIR / name
    df.to_csv(path, index=False)
    print("Saved:", path)
    return path


def to_markdown_table(df):
    if df is None or len(df) == 0:
        return "_No rows._"
    try:
        return df.to_markdown(index=False)
    except Exception:
        return "```text\n" + df.to_string(index=False) + "\n```"

### 5. Query DP1 Butler products

This cell queries the key Butler dataset types for the DP1 ECDFS run.

For this run, VIRCAM coadd images are queried from the Butler collection, while ComCam coadd images are inventoried separately from the FITS directory in the next section.

In [5]:
vircam_calexp_refs, vircam_calexp_df = query_dataset_refs(
    "deepCoadd_calexp",
    [VIRCAM_CALEXP_COLLECTION],
    "VIRCAM coadd images",
)

forced_refs, forced_df = query_dataset_refs(
    "deepCoadd_forced_src",
    [FORCED_COLLECTION],
    "forced catalogues",
)

meas_refs, meas_df = query_dataset_refs(
    "deepCoadd_meas",
    [MEAS_COLLECTION],
    "measured catalogues",
)

vircam_calexp_df = clean_inventory_df(vircam_calexp_df)
forced_df = clean_inventory_df(forced_df)
meas_df = clean_inventory_df(meas_df)

print("VIRCAM deepCoadd_calexp refs:", len(vircam_calexp_df))
print("deepCoadd_forced_src refs:", len(forced_df))
print("deepCoadd_meas refs:", len(meas_df))

display(vircam_calexp_df.head())
display(forced_df.head())
display(meas_df.head())

VIRCAM deepCoadd_calexp refs: 406
deepCoadd_forced_src refs: 837
deepCoadd_meas refs: 837


,source,storage,dataset_type,run,skymap,tract,patch,band,physical_filter
169,VIRCAM coadd images,Butler,deepCoadd_calexp,u/ir-sare1/DRP/videoCoaddDetect/20260405T092826Z,lsst_cells_v1,4848,60,H,None
344,VIRCAM coadd images,Butler,deepCoadd_calexp,u/ir-sare1/DRP/videoCoaddDetect/20260405T092826Z,lsst_cells_v1,4848,60,J,None
18,VIRCAM coadd images,Butler,deepCoadd_calexp,u/ir-sare1/DRP/videoCoaddDetect/20260405T092826Z,lsst_cells_v1,4848,60,K,None
202,VIRCAM coadd images,Butler,deepCoadd_calexp,u/ir-sare1/DRP/videoCoaddDetect/20260405T092826Z,lsst_cells_v1,4848,60,Y,None
120,VIRCAM coadd images,Butler,deepCoadd_calexp,u/ir-sare1/DRP/videoCoaddDetect/20260405T092826Z,lsst_cells_v1,4848,61,H,None


,source,storage,dataset_type,run,skymap,tract,patch,band,physical_filter
459,forced catalogues,Butler,deepCoadd_forced_src,u/ir-sare1/DRP/videoMultiVisit/20260406T094204Z,lsst_cells_v1,4848,60,H,None
588,forced catalogues,Butler,deepCoadd_forced_src,u/ir-sare1/DRP/videoMultiVisit/20260406T094204Z,lsst_cells_v1,4848,60,J,None
421,forced catalogues,Butler,deepCoadd_forced_src,u/ir-sare1/DRP/videoMultiVisit/20260406T094204Z,lsst_cells_v1,4848,60,K,None
30,forced catalogues,Butler,deepCoadd_forced_src,u/ir-sare1/DRP/videoMultiVisit/20260406T094204Z,lsst_cells_v1,4848,60,Y,None
303,forced catalogues,Butler,deepCoadd_forced_src,u/ir-sare1/DRP/videoMultiVisit/20260406T094204Z,lsst_cells_v1,4848,60,g,None


,source,storage,dataset_type,run,skymap,tract,patch,band,physical_filter
312,measured catalogues,Butler,deepCoadd_meas,u/ir-sare1/DRP/videoMultiVisit/20260406T094204Z,lsst_cells_v1,4848,60,H,None
793,measured catalogues,Butler,deepCoadd_meas,u/ir-sare1/DRP/videoMultiVisit/20260406T094204Z,lsst_cells_v1,4848,60,J,None
242,measured catalogues,Butler,deepCoadd_meas,u/ir-sare1/DRP/videoMultiVisit/20260406T094204Z,lsst_cells_v1,4848,60,K,None
287,measured catalogues,Butler,deepCoadd_meas,u/ir-sare1/DRP/videoMultiVisit/20260406T094204Z,lsst_cells_v1,4848,60,Y,None
244,measured catalogues,Butler,deepCoadd_meas,u/ir-sare1/DRP/videoMultiVisit/20260406T094204Z,lsst_cells_v1,4848,60,g,None


### 6. Inventory ComCam coadd FITS images

The DP1 ComCam coadd images are stored as FITS products under:

```text
data/ComCam/deepCoadd_results/deepCoadd_calexp
```

This cell scans that directory and tries to parse the band, tract, and patch from the file path. If some rows are not parsed correctly, inspect the saved CSV and adjust `parse_comcam_metadata_from_path`.

In [6]:
def parse_comcam_metadata_from_path(path):
    path = Path(path)
    parts = list(path.parts)
    s = str(path)
    slow = s.lower()

    band = None
    for part in parts:
        if part in COMCAM_BANDS:
            band = part
            break

    if band is None:
        for b in COMCAM_BANDS:
            patterns = [
                rf"band[-_=]{b}\b",
                rf"filter[-_=]{b}\b",
                rf"\b{b}\b",
                rf"[_\-]{b}[_\-]",
            ]
            if any(re.search(pat, slow) for pat in patterns):
                band = b
                break

    tract = None
    patch = None

    # Prefer explicit labels if present.
    tract_match = re.search(r"tract[-_=]?(\d+)", slow)
    patch_match = re.search(r"patch[-_=]?(\d+)", slow)

    if tract_match:
        tract = int(tract_match.group(1))
    if patch_match:
        patch = int(patch_match.group(1))

    # Otherwise infer from directory parts: first 4-digit number is usually tract;
    # a nearby smaller integer is usually patch.
    numeric_parts = []
    for i, part in enumerate(parts):
        if re.fullmatch(r"\d+", part):
            numeric_parts.append((i, int(part)))

    if tract is None:
        for i, val in numeric_parts:
            if 4000 <= val <= 6000:
                tract = val
                tract_part_index = i
                break
        else:
            tract_part_index = None
    else:
        tract_part_index = None

    if patch is None:
        if tract_part_index is not None:
            for i, val in numeric_parts:
                if i > tract_part_index and 0 <= val <= 999:
                    patch = val
                    break

    # Last fallback: use numbers from filename/path.
    nums = [int(x) for x in re.findall(r"\d+", s)]
    if tract is None:
        for val in nums:
            if 4000 <= val <= 6000:
                tract = val
                break

    if patch is None and tract is not None:
        # Use the first small number after the tract in the full path string.
        split_after_tract = s.split(str(tract), 1)
        if len(split_after_tract) == 2:
            for val in [int(x) for x in re.findall(r"\d+", split_after_tract[1])]:
                if 0 <= val <= 999:
                    patch = val
                    break

    return {
        "source": "ComCam coadd images",
        "storage": "FITS",
        "dataset_type": "deepCoadd_calexp",
        "run": str(COMCAM_CALEXP_DIR),
        "skymap": SKYMAP,
        "tract": tract,
        "patch": patch,
        "band": band,
        "physical_filter": band,
        "path": str(path),
    }


def scan_comcam_calexp_directory():
    if not COMCAM_CALEXP_DIR.exists():
        print("ComCam coadd directory does not exist:", COMCAM_CALEXP_DIR)
        return pd.DataFrame()

    files = (
        list(COMCAM_CALEXP_DIR.rglob("*.fits"))
        + list(COMCAM_CALEXP_DIR.rglob("*.fits.fz"))
        + list(COMCAM_CALEXP_DIR.rglob("*.fz"))
    )

    rows = [parse_comcam_metadata_from_path(p) for p in files]
    df = pd.DataFrame(rows)

    if len(df) > 0:
        for col in ["tract", "patch"]:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    return df.sort_values(["tract", "patch", "band", "path"], na_position="last")


comcam_calexp_df = scan_comcam_calexp_directory()

print("ComCam FITS coadd files:", len(comcam_calexp_df))
if len(comcam_calexp_df) > 0:
    print("Rows with parsed band:", int(comcam_calexp_df["band"].notna().sum()))
    print("Rows with parsed tract:", int(comcam_calexp_df["tract"].notna().sum()))
    print("Rows with parsed patch:", int(comcam_calexp_df["patch"].notna().sum()))

save_table(comcam_calexp_df, "dp1_comcam_calexp_files.csv")
display(comcam_calexp_df.head(20))

ComCam FITS coadd files: 485
Rows with parsed band: 485
Rows with parsed tract: 485
Rows with parsed patch: 485
Saved: data/dp1_run_inventory/dp1_comcam_calexp_files.csv


,source,storage,dataset_type,run,skymap,tract,patch,band,physical_filter,path
435,ComCam coadd images,FITS,deepCoadd_calexp,../../../dmu4/dmu4_DP1/dmu4_DP1_ECDFS/data/Com...,lsst_cells_v1,4848,60,g,g,../../../dmu4/dmu4_DP1/dmu4_DP1_ECDFS/data/Com...
440,ComCam coadd images,FITS,deepCoadd_calexp,../../../dmu4/dmu4_DP1/dmu4_DP1_ECDFS/data/Com...,lsst_cells_v1,4848,60,i,i,../../../dmu4/dmu4_DP1/dmu4_DP1_ECDFS/data/Com...
439,ComCam coadd images,FITS,deepCoadd_calexp,../../../dmu4/dmu4_DP1/dmu4_DP1_ECDFS/data/Com...,lsst_cells_v1,4848,60,r,r,../../../dmu4/dmu4_DP1/dmu4_DP1_ECDFS/data/Com...
438,ComCam coadd images,FITS,deepCoadd_calexp,../../../dmu4/dmu4_DP1/dmu4_DP1_ECDFS/data/Com...,lsst_cells_v1,4848,60,u,u,../../../dmu4/dmu4_DP1/dmu4_DP1_ECDFS/data/Com...
437,ComCam coadd images,FITS,deepCoadd_calexp,../../../dmu4/dmu4_DP1/dmu4_DP1_ECDFS/data/Com...,lsst_cells_v1,4848,60,y,y,../../../dmu4/dmu4_DP1/dmu4_DP1_ECDFS/data/Com...
436,ComCam coadd images,FITS,deepCoadd_calexp,../../../dmu4/dmu4_DP1/dmu4_DP1_ECDFS/data/Com...,lsst_cells_v1,4848,60,z,z,../../../dmu4/dmu4_DP1/dmu4_DP1_ECDFS/data/Com...
458,ComCam coadd images,FITS,deepCoadd_calexp,../../../dmu4/dmu4_DP1/dmu4_DP1_ECDFS/data/Com...,lsst_cells_v1,4848,61,g,g,../../../dmu4/dmu4_DP1/dmu4_DP1_ECDFS/data/Com...
461,ComCam coadd images,FITS,deepCoadd_calexp,../../../dmu4/dmu4_DP1/dmu4_DP1_ECDFS/data/Com...,lsst_cells_v1,4848,61,i,i,../../../dmu4/dmu4_DP1/dmu4_DP1_ECDFS/data/Com...
460,ComCam coadd images,FITS,deepCoadd_calexp,../../../dmu4/dmu4_DP1/dmu4_DP1_ECDFS/data/Com...,lsst_cells_v1,4848,61,y,y,../../../dmu4/dmu4_DP1/dmu4_DP1_ECDFS/data/Com...
459,ComCam coadd images,FITS,deepCoadd_calexp,../../../dmu4/dmu4_DP1/dmu4_DP1_ECDFS/data/Com...,lsst_cells_v1,4848,61,z,z,../../../dmu4/dmu4_DP1/dmu4_DP1_ECDFS/data/Com...


### 7. Product footprint tables

This section creates the footprint tables needed for the DP1 run documentation:

- ComCam coadd-image patch counts per tract and band.
- VIRCAM coadd-image patch counts per tract and band.
- Forced-catalogue patch counts per tract and band.
- Measured-catalogue patch counts, if available.

In [7]:
comcam_counts = count_by_tract_band(comcam_calexp_df)
vircam_counts = count_by_tract_band(vircam_calexp_df)
forced_counts = count_by_tract_band(forced_df)
meas_counts = count_by_tract_band(meas_df)

print("ComCam coadd-image footprint")
display(comcam_counts)

print("VIRCAM coadd-image footprint")
display(vircam_counts)

print("Forced-catalogue footprint")
display(forced_counts)

print("Measured-catalogue footprint")
display(meas_counts)

save_table(comcam_counts, "dp1_comcam_calexp_counts_by_tract_band.csv")
save_table(vircam_counts, "dp1_vircam_calexp_counts_by_tract_band.csv")
save_table(forced_counts, "dp1_forced_counts_by_tract_band.csv")
save_table(meas_counts, "dp1_meas_counts_by_tract_band.csv")

ComCam coadd-image footprint


band,tract,u,g,r,i,z,y
0,4848,12,13,12,14,13,12
1,4849,18,19,19,19,20,17
2,5062,0,1,0,1,3,0
3,5063,42,47,47,48,46,43
4,5064,2,3,4,4,4,2


VIRCAM coadd-image footprint


band,tract,Z,Y,J,H,K
0,4848,4,14,14,14,14
1,4849,3,20,20,20,20
2,5062,3,3,3,3,3
3,5063,32,50,50,50,50
4,5064,0,4,4,4,4


Forced-catalogue footprint


band,tract,u,g,r,i,z,y,Z,Y,J,H,K
0,4848,11,12,11,13,12,11,3,13,13,13,13
1,4849,17,18,18,18,19,16,2,19,19,19,19
2,5062,0,1,0,1,3,0,3,3,3,3,3
3,5063,39,44,44,45,43,40,30,47,47,47,47
4,5064,2,3,4,4,4,2,0,4,4,4,4


Measured-catalogue footprint


band,tract,u,g,r,i,z,y,Z,Y,J,H,K
0,4848,11,12,11,13,12,11,3,13,13,13,13
1,4849,17,18,18,18,19,16,2,19,19,19,19
2,5062,0,1,0,1,3,0,3,3,3,3,3
3,5063,39,44,44,45,43,40,30,47,47,47,47
4,5064,2,3,4,4,4,2,0,4,4,4,4


Saved: data/dp1_run_inventory/dp1_comcam_calexp_counts_by_tract_band.csv
Saved: data/dp1_run_inventory/dp1_vircam_calexp_counts_by_tract_band.csv
Saved: data/dp1_run_inventory/dp1_forced_counts_by_tract_band.csv
Saved: data/dp1_run_inventory/dp1_meas_counts_by_tract_band.csv


PosixPath('data/dp1_run_inventory/dp1_meas_counts_by_tract_band.csv')

### 8. Patch-unit summaries

Patch-unit summaries count distinct `(tract, patch)` units, regardless of band. These are useful for a short run note because they describe the spatial footprint compactly.

In [8]:
comcam_patch_units = patch_units_by_tract(comcam_calexp_df)
vircam_patch_units = patch_units_by_tract(vircam_calexp_df)
forced_patch_units = patch_units_by_tract(forced_df)
meas_patch_units = patch_units_by_tract(meas_df)

print("ComCam coadd-image patch units")
display(comcam_patch_units)

print("VIRCAM coadd-image patch units")
display(vircam_patch_units)

print("Forced-catalogue patch units")
display(forced_patch_units)

print("Measured-catalogue patch units")
display(meas_patch_units)

save_table(comcam_patch_units, "dp1_comcam_calexp_patch_units.csv")
save_table(vircam_patch_units, "dp1_vircam_calexp_patch_units.csv")
save_table(forced_patch_units, "dp1_forced_patch_units.csv")
save_table(meas_patch_units, "dp1_meas_patch_units.csv")

ComCam coadd-image patch units


,tract,n_patches
0,4848,14
1,4849,20
2,5062,3
3,5063,50
4,5064,4
5,Total,91


VIRCAM coadd-image patch units


,tract,n_patches
0,4848,14
1,4849,20
2,5062,3
3,5063,50
4,5064,4
5,Total,91


Forced-catalogue patch units


,tract,n_patches
0,4848,13
1,4849,19
2,5062,3
3,5063,47
4,5064,4
5,Total,86


Measured-catalogue patch units


,tract,n_patches
0,4848,13
1,4849,19
2,5062,3
3,5063,47
4,5064,4
5,Total,86


Saved: data/dp1_run_inventory/dp1_comcam_calexp_patch_units.csv
Saved: data/dp1_run_inventory/dp1_vircam_calexp_patch_units.csv
Saved: data/dp1_run_inventory/dp1_forced_patch_units.csv
Saved: data/dp1_run_inventory/dp1_meas_patch_units.csv


PosixPath('data/dp1_run_inventory/dp1_meas_patch_units.csv')

### 9. Combined product inventory and missing-product checks

This cell combines the ComCam and VIRCAM coadd-image footprints and compares them with the forced-catalogue footprint. The output lists cases where a coadd image exists but the corresponding forced catalogue is missing, and cases where a forced catalogue exists without a parsed coadd-image entry.

In [9]:
coadd_df = pd.concat(
    [
        comcam_calexp_df[["source", "storage", "dataset_type", "run", "skymap", "tract", "patch", "band", "physical_filter"]]
        if len(comcam_calexp_df) > 0 else pd.DataFrame(),
        vircam_calexp_df,
    ],
    ignore_index=True,
)

product_inventory = pd.concat(
    [
        coadd_df,
        forced_df,
        meas_df,
    ],
    ignore_index=True,
)

product_inventory = clean_inventory_df(product_inventory)

save_table(product_inventory, "dp1_product_inventory_all_refs.csv")

def combo_set(df):
    if df is None or len(df) == 0:
        return set()
    use = df.dropna(subset=["tract", "patch", "band"]).copy()
    return set(
        (
            int(row.tract),
            int(row.patch),
            str(row.band),
        )
        for row in use.itertuples(index=False)
    )

coadd_combos = combo_set(coadd_df)
forced_combos = combo_set(forced_df)

coadd_without_forced = sorted(coadd_combos - forced_combos)
forced_without_coadd = sorted(forced_combos - coadd_combos)

missing_rows = []

for tract, patch, band in coadd_without_forced:
    missing_rows.append({
        "issue": "coadd_exists_forced_missing",
        "tract": tract,
        "patch": patch,
        "band": band,
    })

for tract, patch, band in forced_without_coadd:
    missing_rows.append({
        "issue": "forced_exists_coadd_not_found",
        "tract": tract,
        "patch": patch,
        "band": band,
    })

missing_df = pd.DataFrame(missing_rows)
if len(missing_df) > 0:
    missing_df = missing_df.sort_values(["issue", "tract", "patch", "band"])

print("N coadd combinations:", len(coadd_combos))
print("N forced combinations:", len(forced_combos))
print("N coadd exists but forced missing:", len(coadd_without_forced))
print("N forced exists but coadd not found:", len(forced_without_coadd))

save_table(missing_df, "dp1_missing_product_checks.csv")
display(missing_df.head(50))

Saved: data/dp1_run_inventory/dp1_product_inventory_all_refs.csv
N coadd combinations: 891
N forced combinations: 837
N coadd exists but forced missing: 54
N forced exists but coadd not found: 0
Saved: data/dp1_run_inventory/dp1_missing_product_checks.csv


,issue,tract,patch,band
0,coadd_exists_forced_missing,4848,90,H
1,coadd_exists_forced_missing,4848,90,J
2,coadd_exists_forced_missing,4848,90,K
3,coadd_exists_forced_missing,4848,90,Y
4,coadd_exists_forced_missing,4848,90,Z
5,coadd_exists_forced_missing,4848,90,g
6,coadd_exists_forced_missing,4848,90,i
7,coadd_exists_forced_missing,4848,90,r
8,coadd_exists_forced_missing,4848,90,u
9,coadd_exists_forced_missing,4848,90,y


### 10. Forced-catalogue row counts

This cell loads each `deepCoadd_forced_src` dataset and counts the number of rows. It is useful for identifying empty or very low-row catalogues.

If this is slow, set `COUNT_FORCED_ROWS = False` in the settings cell.

In [10]:
def count_rows_for_forced_refs(refs):
    rows = []

    for i, ref in enumerate(refs):
        row = ref_to_row(ref, "deepCoadd_forced_src", "forced catalogues")

        try:
            cat = butler.get(ref)
            row["n_rows"] = len(cat)
            row["status"] = "ok"
        except Exception as e:
            row["n_rows"] = np.nan
            row["status"] = f"ERROR: {type(e).__name__}: {e}"

        rows.append(row)

        if (i + 1) % 25 == 0 or (i + 1) == len(refs):
            print(f"Counted {i + 1}/{len(refs)} forced catalogues")

    out = pd.DataFrame(rows)

    if len(out) > 0:
        for col in ["tract", "patch"]:
            out[col] = pd.to_numeric(out[col], errors="coerce").astype("Int64")

    return out


if COUNT_FORCED_ROWS:
    forced_row_counts = count_rows_for_forced_refs(forced_refs)
else:
    forced_row_counts = pd.DataFrame()

if len(forced_row_counts) > 0:
    low_row_forced = forced_row_counts[
        (forced_row_counts["status"] != "ok")
        | (forced_row_counts["n_rows"].fillna(-1) < LOW_ROW_THRESHOLD)
    ].copy()

    print("Forced catalogue row-count summary")
    display(forced_row_counts["n_rows"].describe())

    print(f"Forced catalogues with fewer than {LOW_ROW_THRESHOLD} rows, or read errors")
    display(low_row_forced.sort_values(["tract", "patch", "band"]).head(100))
else:
    low_row_forced = pd.DataFrame()
    print("Forced row counting skipped.")

save_table(forced_row_counts, "dp1_forced_row_counts.csv")
save_table(low_row_forced, "dp1_forced_low_row_or_error_catalogues.csv")

Forced row counting skipped.
Saved: data/dp1_run_inventory/dp1_forced_row_counts.csv
Saved: data/dp1_run_inventory/dp1_forced_low_row_or_error_catalogues.csv


PosixPath('data/dp1_run_inventory/dp1_forced_low_row_or_error_catalogues.csv')

### 11. Final reduced catalogue summary

This cell summarises the final reduced catalogue used by the photometry and astrometry notebooks.

In [11]:
final_catalogue_summary = {}

if FINAL_REDUCED_CATALOGUE.exists() and Table is not None:
    tab = Table.read(FINAL_REDUCED_CATALOGUE)

    final_catalogue_summary = {
        "path": str(FINAL_REDUCED_CATALOGUE),
        "n_rows": len(tab),
        "n_columns": len(tab.colnames),
    }

    possible_ra_cols = [
        "VIRCAM_K_m_coord_ra",
        "ComCam_r_m_coord_ra",
        "coord_ra",
        "ra",
    ]

    possible_dec_cols = [
        "VIRCAM_K_m_coord_dec",
        "ComCam_r_m_coord_dec",
        "coord_dec",
        "dec",
    ]

    ra_col = next((c for c in possible_ra_cols if c in tab.colnames), None)
    dec_col = next((c for c in possible_dec_cols if c in tab.colnames), None)

    if ra_col is not None and dec_col is not None:
        ra = np.asarray(tab[ra_col], float)
        dec = np.asarray(tab[dec_col], float)

        # Most LSST coord columns are radians; convert if needed.
        if np.nanmax(np.abs(ra)) <= 2 * np.pi + 0.1:
            ra = np.rad2deg(ra)
        if np.nanmax(np.abs(dec)) <= np.pi + 0.1:
            dec = np.rad2deg(dec)

        final_catalogue_summary.update({
            "ra_col": ra_col,
            "dec_col": dec_col,
            "ra_min_deg": float(np.nanmin(ra)),
            "ra_max_deg": float(np.nanmax(ra)),
            "dec_min_deg": float(np.nanmin(dec)),
            "dec_max_deg": float(np.nanmax(dec)),
        })

    print("Final reduced catalogue summary:")
    for key, value in final_catalogue_summary.items():
        print(f"  {key}: {value}")

    col_summary = pd.DataFrame({"column": tab.colnames})
    save_table(col_summary, "dp1_final_reduced_catalogue_columns.csv")

else:
    print("Final reduced catalogue not found or astropy is unavailable:", FINAL_REDUCED_CATALOGUE)

summary_json_path = OUTDIR / "dp1_final_reduced_catalogue_summary.json"
summary_json_path.write_text(json.dumps(final_catalogue_summary, indent=2), encoding="utf-8")
print("Saved:", summary_json_path)

Final reduced catalogue summary:
  path: ../data/full_reduced_cat_DP1_20260408.fits
  n_rows: 1380173
  n_columns: 87
  ra_col: VIRCAM_K_m_coord_ra
  dec_col: VIRCAM_K_m_coord_dec
  ra_min_deg: 52.17037090129214
  ra_max_deg: 54.046426717445286
  dec_min_deg: -28.851470987730547
  dec_max_deg: -27.34205997261303
Saved: data/dp1_run_inventory/dp1_final_reduced_catalogue_columns.csv
Saved: data/dp1_run_inventory/dp1_final_reduced_catalogue_summary.json


### 12. Write a compact Markdown summary for the DP1 run note

After running the notebook, share the generated Markdown file and CSV tables. They contain the information needed to write the final DP1 run documentation.

In [13]:
def total_refs(df):
    return int(len(df)) if df is not None else 0


def distinct_patch_units_count(df):
    if df is None or len(df) == 0:
        return 0
    use = df.dropna(subset=["tract", "patch"]).copy()
    if len(use) == 0:
        return 0
    return int(use.drop_duplicates(["tract", "patch"]).shape[0])


def maybe_markdown_section(title, df):
    return f"#### {title}\n\n{to_markdown_table(df)}\n"


lines = []

lines.append("# DP1 ECDFS run inventory summary")
lines.append("")
lines.append(f"Generated: {datetime.now().isoformat(timespec='seconds')}")
lines.append("")
lines.append("## 1. Run locations and collections")
lines.append("")
lines.append(f"- Butler repo: `{BUTLER_REPO}`")
lines.append(f"- Skymap: `{SKYMAP}`")
lines.append(f"- VIRCAM coadd images (`deepCoadd_calexp`): `{VIRCAM_CALEXP_COLLECTION}`")
lines.append(f"- ComCam coadd images (`deepCoadd_calexp` FITS products): `{COMCAM_CALEXP_DIR}`")
lines.append(f"- Forced catalogues (`deepCoadd_forced_src`): `{FORCED_COLLECTION}`")
lines.append(f"- Measured catalogues (`deepCoadd_meas`, if present): `{MEAS_COLLECTION}`")
lines.append(f"- Final reduced catalogue: `{FINAL_REDUCED_CATALOGUE}`")
lines.append("")
lines.append("## 2. Product totals")
lines.append("")
lines.append(f"- ComCam coadd FITS files: **{total_refs(comcam_calexp_df)}**")
lines.append(f"- VIRCAM `deepCoadd_calexp` refs: **{total_refs(vircam_calexp_df)}**")
lines.append(f"- `deepCoadd_forced_src` refs: **{total_refs(forced_df)}**")
lines.append(f"- `deepCoadd_meas` refs: **{total_refs(meas_df)}**")
lines.append(f"- ComCam coadd patch units: **{distinct_patch_units_count(comcam_calexp_df)}**")
lines.append(f"- VIRCAM coadd patch units: **{distinct_patch_units_count(vircam_calexp_df)}**")
lines.append(f"- Forced-catalogue patch units: **{distinct_patch_units_count(forced_df)}**")
lines.append("")
lines.append("## 3. Footprint tables")
lines.append("")
lines.append(maybe_markdown_section("ComCam coadd-image footprint", comcam_counts))
lines.append(maybe_markdown_section("VIRCAM coadd-image footprint", vircam_counts))
lines.append(maybe_markdown_section("Forced-catalogue footprint", forced_counts))
lines.append(maybe_markdown_section("Measured-catalogue footprint", meas_counts))
lines.append("")
lines.append("## 4. Patch-unit summaries")
lines.append("")
lines.append(maybe_markdown_section("ComCam coadd patch units", comcam_patch_units))
lines.append(maybe_markdown_section("VIRCAM coadd patch units", vircam_patch_units))
lines.append(maybe_markdown_section("Forced-catalogue patch units", forced_patch_units))
lines.append("")
lines.append("## 5. Missing or incomplete products")
lines.append("")
lines.append(f"- Coadd combinations without forced catalogue: **{len(coadd_without_forced)}**")
lines.append(f"- Forced combinations without parsed coadd image: **{len(forced_without_coadd)}**")
lines.append(f"- Forced catalogues with fewer than {LOW_ROW_THRESHOLD} rows or read errors: **{len(low_row_forced)}**")
lines.append("")
if len(missing_df) > 0:
    lines.append("First missing-product rows:")
    lines.append("")
    lines.append(to_markdown_table(missing_df.head(30)))
    lines.append("")
if len(low_row_forced) > 0:
    lines.append("First low-row/error forced-catalogue rows:")
    lines.append("")
    lines.append(to_markdown_table(low_row_forced.head(30)))
    lines.append("")
lines.append("## 6. Final reduced catalogue")
lines.append("")
if final_catalogue_summary:
    for key, value in final_catalogue_summary.items():
        lines.append(f"- {key}: `{value}`")
else:
    lines.append("_Final reduced catalogue summary was not available._")
lines.append("")
lines.append("## 7. Suggested documentation sentence")
lines.append("")
lines.append(
    "The DP1 ECDFS run provides a completed early Rubin/ComCam validation case "
    "for the LSST--VISTA workflow, combining ComCam optical coadds with VIRCAM "
    "near-infrared coadds and forced catalogues in the ECDFS field."
)

summary_md = OUTDIR / "dp1_run_summary_for_report.md"
summary_md.write_text("\n".join(lines), encoding="utf-8")

print("Saved Markdown summary:", summary_md)

Saved Markdown summary: data/dp1_run_inventory/dp1_run_summary_for_report.md
